# Decomposition & stationarity — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def make_series(n=60, trend=0.08, amp=2.0, noise=0.25):
    t=np.arange(n)
    return t, 10 + trend*t + amp*np.sin(2*np.pi*t/12) + noise*np.random.randn(n)
def acf(x, max_lag):
    x=np.asarray(x)-np.mean(x); den=np.dot(x,x)
    return np.array([1.0]+[np.dot(x[:-k],x[k:])/den for k in range(1,max_lag+1)])
def moving_average(x,w):
    return np.convolve(x, np.ones(w)/w, mode='valid')
print('setup ok')

## Separating a time series into trend, seasonality, residuals, then asking whether the residual rules stay stable
Decomposition says observed values are built from reusable pieces. These five examples show the synthetic trend-season-noise series, recover its trend, read the seasonal residual, test a stationarity failure, and fix it by differencing.

In [ ]:
# 1) additive series: value = level + trend + season + noise
t=np.arange(48); trend=0.08*t; season=2*np.sin(2*np.pi*t/12); noise=0.15*np.random.randn(48); y=10+trend+season+noise
plt.figure(figsize=(6,3)); plt.plot(t,y,label='observed'); plt.plot(t,10+trend,label='trend'); plt.legend(); plt.title('observed = trend + season + noise')
assert y.shape==(48,) and abs(y[0]-10.264607851895657)<1e-9

In [ ]:
# 2) least-squares trend: remove the slow drift before reading seasonality
t6=np.arange(6); y6=10+0.5*t6+np.array([0,2,0,-2,0,2]); A=np.vstack([np.ones(6),t6]).T; beta=np.linalg.lstsq(A,y6,rcond=None)[0]
fit=A@beta; resid=y6-fit
plt.figure(figsize=(6,3)); plt.plot(t6,y6,'o-',label='observed'); plt.plot(t6,fit,'--',label='fit'); plt.legend(); plt.title(f'fit intercept={beta[0]:.3f}, slope={beta[1]:.3f}')
assert np.allclose(beta,[10.19047619047619,0.5571428571428572])

In [ ]:
# 3) residuals expose the seasonal pattern after trend removal
t6=np.arange(6); y6=10+0.5*t6+np.array([0,2,0,-2,0,2]); A=np.vstack([np.ones(6),t6]).T; resid=y6-A@np.linalg.lstsq(A,y6,rcond=None)[0]
plt.figure(figsize=(6,3)); plt.axhline(0,color='k',lw=1); plt.stem(t6,resid); plt.title('detrended residuals carry seasonality')
assert np.allclose(np.round(resid,3),[-0.19,1.752,-0.305,-2.362,-0.419,1.524])

In [ ]:
# 4) nonstationarity: rolling means climb when trend remains
t,y=make_series(72, trend=0.12, amp=1.0, noise=0.05); rm=moving_average(y,12)
plt.figure(figsize=(6,3)); plt.plot(y,label='series'); plt.plot(np.arange(11,72),rm,label='12-step rolling mean'); plt.legend(); plt.title('trend makes the mean time-dependent')
assert rm[-1]-rm[0] > 6.0

In [ ]:
# 5) first differencing reduces the drifting mean
t,y=make_series(72, trend=0.12, amp=1.0, noise=0.05); dy=np.diff(y); rm=moving_average(dy,12)
plt.figure(figsize=(6,3)); plt.plot(dy,label='first difference'); plt.plot(np.arange(11,71),rm,label='rolling mean'); plt.legend(); plt.title('differences are closer to stationary')
assert abs(rm[-1]-rm[0]) < 0.2